In [1]:
%matplotlib inline

In [2]:
import sys
print(sys.executable)

/Users/owner/Desktop/Personal_Projects/phase_space_explorer/venv/bin/python


In [12]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, VBox, HBox, Layout
from IPython.display import display, Markdown, Math

In [13]:
def logistic_rhs(x, r, K):
    return r * x * (1 - x / K)


def simulate_logistic(r, K, x0, T, dt=0.01):
    n_steps = int(T / dt) + 1
    t = np.linspace(0, T, n_steps)
    x = np.zeros_like(t)
    x[0] = x0

    for i in range(1, len(t)):
        dxdt = logistic_rhs(x[i - 1], r, K)
        x[i] = x[i - 1] + dt * dxdt

    return t, x


def logistic_equilibria(K):
    return [0.0, K]


def logistic_derivative(x, r, K):
    return r * (1 - 2 * x / K)


def classify_equilibrium(x_star, r, K, tol=1e-10):
    val = logistic_derivative(x_star, r, K)
    if val < -tol:
        return "stable"
    elif val > tol:
        return "unstable"
    else:
        return "non-hyperbolic"

In [14]:
def long_term_behavior(x0, r, K):
    if K <= 0:
        return "This model is not in its usual physical regime because K <= 0."

    if r > 0:
        if x0 < 0:
            return "Initial condition is negative, which is mathematically allowed here but usually not physically meaningful for population."
        elif np.isclose(x0, 0):
            return "The system starts at the equilibrium x = 0."
        elif np.isclose(x0, K):
            return "The system starts at the equilibrium x = K."
        elif x0 < K:
            return "The state increases toward the carrying capacity K."
        else:
            return "The state decreases toward the carrying capacity K."

    elif r < 0:
        if np.isclose(x0, 0):
            return "The system starts at the equilibrium x = 0."
        elif np.isclose(x0, K):
            return "The system starts at the equilibrium x = K."
        elif 0 < x0 < K:
            return "The state decreases toward x = 0."
        else:
            return "With r < 0, the qualitative behavior is reversed from the usual logistic-growth regime."

    else:
        return "Since r = 0, the system does not evolve: dx/dt = 0 for all x."

In [15]:
def plot_trajectory(ax, t, x, K):
    ax.plot(t, x, linewidth=2)
    ax.axhline(K, linestyle="--", linewidth=1, label="K")
    ax.set_title("Trajectory: $x(t)$")
    ax.set_xlabel("t")
    ax.set_ylabel("x(t)")
    ax.grid(True, alpha=0.3)
    ax.legend()


def plot_rhs(ax, r, K, x_min, x_max):
    xs = np.linspace(x_min, x_max, 500)
    fx = logistic_rhs(xs, r, K)
    ax.plot(xs, fx, linewidth=2)
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_title("Vector Field Function: $f(x)=rx(1-x/K)$")
    ax.set_xlabel("x")
    ax.set_ylabel("$dx/dt$")
    ax.grid(True, alpha=0.3)


def plot_phase_line(ax, r, K, x_min, x_max):
    ax.set_title("Phase Line")
    ax.set_xlim(-1, 1)
    ax.set_ylim(x_min, x_max)
    ax.set_xticks([])
    ax.set_ylabel("x")
    ax.plot([0, 0], [x_min, x_max], linewidth=2)

    eq_points = logistic_equilibria(K)

    # mark equilibria
    for x_star in eq_points:
        stability = classify_equilibrium(x_star, r, K)
        if stability == "stable":
            ax.plot(0, x_star, "o", markersize=8)
        else:
            ax.plot(0, x_star, "o", markersize=8, markerfacecolor="white")

    # arrows showing sign of dx/dt
    sample_points = np.linspace(x_min, x_max, 9)
    for y in sample_points:
        if np.any(np.isclose(y, eq_points, atol=0.05 * max(1, abs(x_max - x_min)))):
            continue

        val = logistic_rhs(y, r, K)
        if abs(val) < 1e-12:
            continue

        dy = 0.35 * (x_max - x_min) / 10
        if val > 0:
            ax.arrow(0, y - dy, 0, 2 * dy, head_width=0.08, head_length=0.04 * (x_max - x_min),
                     length_includes_head=True)
        else:
            ax.arrow(0, y + dy, 0, -2 * dy, head_width=0.08, head_length=0.04 * (x_max - x_min),
                     length_includes_head=True)

    ax.grid(True, axis="y", alpha=0.2)

In [16]:
def equilibrium_summary_markdown(r, K):
    eqs = logistic_equilibria(K)

    lines = []
    lines.append("### Equilibria and Stability")
    lines.append("")
    lines.append("| Equilibrium | Derivative Test | Classification |")
    lines.append("|---|---:|---|")

    for x_star in eqs:
        deriv = logistic_derivative(x_star, r, K)
        cls = classify_equilibrium(x_star, r, K)
        lines.append(f"| $x^* = {x_star:.3g}$ | ${deriv:.3g}$ | {cls} |")

    lines.append("")
    lines.append("For a 1D autonomous system, an equilibrium is:")
    lines.append("- **stable** if $f'(x^*) < 0$")
    lines.append("- **unstable** if $f'(x^*) > 0$")
    lines.append("- **non-hyperbolic** if $f'(x^*) = 0$")

    return "\n".join(lines)

In [19]:
def logistic_dashboard(r=1.0, K=10.0, x0=1.0, T=10, dt=0.01):
    if K == 0:
        display(Markdown("## Invalid choice: $K=0$ makes the equation singular."))
        return

    t, x = simulate_logistic(r=r, K=K, x0=x0, T=T, dt=dt)

    x_min_data = min(np.min(x), 0, x0, K)
    x_max_data = max(np.max(x), x0, K)

    pad = 0.15 * max(1.0, x_max_data - x_min_data)
    x_min = x_min_data - pad
    x_max = x_max_data + pad

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

    plot_trajectory(axes[0], t, x, K)
    plot_rhs(axes[1], r, K, x_min, x_max)
    plot_phase_line(axes[2], r, K, x_min, x_max)

    plt.tight_layout()
    plt.show()

    interpretation = long_term_behavior(x0, r, K)

    # Equation
    display(Markdown("## Logistic Equation"))
    display(Math(r"\frac{dx}{dt} = r x \left(1-\frac{x}{K}\right)"))

    # Parameter values
    display(Markdown("### Current Parameters"))
    display(Math(fr"r={r},\quad K={K},\quad x_0={x0},\quad T={T}"))

    # Interpretation
    display(Markdown("### Interpretation"))
    display(Markdown(interpretation))

    # Stability table
    display(Markdown(equilibrium_summary_markdown(r, K)))

In [20]:
interact(
    logistic_dashboard,
    r=FloatSlider(value=1.0, min=-2.0, max=3.0, step=0.05, description='r'),
    K=FloatSlider(value=10.0, min=0.5, max=20.0, step=0.5, description='K'),
    x0=FloatSlider(value=1.0, min=-5.0, max=25.0, step=0.25, description='x₀'),
    T=IntSlider(value=10, min=1, max=40, step=1, description='T'),
    dt=FloatSlider(value=0.01, min=0.005, max=0.1, step=0.005, description='dt')
);

interactive(children=(FloatSlider(value=1.0, description='r', max=3.0, min=-2.0, step=0.05), FloatSlider(value…